# Challenge 12: BTS 콘서트 티켓 예매 시스템

**난이도:** ★★★★

**사전 요구사항:**
- 노트북 02: Redis Streams 기초
- 노트북 03: RabbitMQ Fanout Exchange

**네비게이션:** [← 이전 (Challenge 11)](./11_challenge_coupang_order.ipynb) | [다음 (Challenge 13) →](./13_challenge_kakaotalk_chat.ipynb)

## 시나리오: BTS 월드투어 서울 콘서트 티켓팅

**상황:**
- BTS 월드투어 서울 콘서트 티켓 오픈
- 총 좌석 수: **50석** (VIP 10석, R석 15석, S석 15석, A석 10석)
- 동시 예매 시도: **200명**
- 경쟁률: **4:1**

### 왜 이 패턴을 사용하는가?

| 문제 | 해결 패턴 | 이유 |
|------|-----------|------|
| 200명이 동시 요청 | **Redis Stream** 큐잉 | 순서 보장 + 영속성, 요청 유실 방지 |
| 대량 요청 빠른 처리 | **Consumer Group** 분산 처리 | 여러 워커가 병렬로 요청 소비 |
| 50석만 정확히 판매 | **Redis Cache** 좌석 관리 | 캐시로 좌석 상태 추적 |
| 예매 성공 알림 | **Fanout Exchange** 브로드캐스트 | 이메일+SMS 동시 알림 |

### 처리 흐름
1. 200건의 예매 요청을 Redis Stream에 추가 (큐잉)
2. Consumer Group의 워커들이 요청을 분산 처리
3. 각 요청마다 Cache에서 좌석 잔여석 확인/차감
4. 예매 성공 시 RabbitMQ Fanout으로 알림 브로드캐스트

## 아키텍처 다이어그램

```text
  200명의 팬
      │
      │  POST /redis/stream/add  (예매 요청 큐잉)
      ▼
┌─────────────────────┐
│   Redis Stream      │   ← 순서 보장, 영속성, 재처리 가능
│   (ticket-booking)  │
└─────────────────────┘
      │
      │  Consumer Group: ticket-workers
      │  GET /redis/stream/group/read
      ▼
┌─────────────────────────────────┐
│   worker-1  │  worker-2  │  worker-3  │   ← 병렬 처리
└─────────────────────────────────┘
      │
      │  각 요청마다 Cache 조회/갱신
      │  GET /redis/cache/get/seats:{section}
      │  POST /redis/cache/set
      ▼
┌──────────────────┐
│  Redis Cache     │   ← 좌석 상태 관리
│  seats:VIP = 10  │      available 카운트 추적
│  seats:R   = 15  │
│  seats:S   = 15  │
│  seats:A   = 10  │
└──────────────────┘
      │
      ├── 성공 (50건) ─────────────────┐
      │                                 ▼
      │                    ┌──────────────────────┐
      │                    │  RabbitMQ Fanout     │
      │                    │  POST /rabbitmq/     │
      │                    │   fanout/publish     │
      │                    └──────────────────────┘
      │                           │
      │                    ┌──────┴──────┐
      │                    ▼             ▼
      │              ┌──────────┐  ┌──────────┐
      │              │  Email   │  │   SMS    │
      │              │  Queue   │  │  Queue   │
      │              └──────────┘  └──────────┘
      │
      └── 실패 (150건) → "매진" 응답
```

In [ ]:
import httpx
import asyncio
import json
import time
from pathlib import Path

BASE_URL = "http://localhost:8000"

# ── 헬스 체크 ──
async with httpx.AsyncClient(timeout=10.0) as client:
    resp = await client.get(f"{BASE_URL}/health")
    assert resp.status_code == 200, f"서버 연결 실패: {resp.status_code}"
    print(f"서버 상태: {resp.json()}")

# ── Mock 데이터 로드 ──
tickets_path = Path("../data/mock/tickets.json")
with open(tickets_path, "r", encoding="utf-8") as f:
    tickets_data = json.load(f)

concert = tickets_data["concert"]
requests_data = tickets_data["requests"]  # 200건의 예매 요청

print(f"\n콘서트: {concert['name']}")
print(f"장소: {concert['venue']}")
print(f"일시: {concert['date']}")
print(f"총 좌석: {concert['total_seats']}석")
print(f"\n좌석 구성:")
for section in concert["sections"]:
    print(f"  - {section['name']}: {section['seats']}석 ({section['price']:,}원)")
print(f"\n예매 요청 수: {len(requests_data)}건")
print(f"경쟁률: {len(requests_data) / concert['total_seats']:.0f}:1")

## Step 1: Redis Stream으로 예매 요청 큐잉

### 왜 Redis Stream인가?

200명이 **동시에** 예매 버튼을 누르면 서버에 요청이 한꺼번에 몰립니다.
이 요청들을 바로 처리하면 서버가 과부하될 수 있습니다.

**Redis Stream의 역할:**
- 요청을 **순서대로 큐에 저장** (선착순 보장)
- 서버가 처리할 수 있는 속도로 **하나씩 꺼내서 처리**
- 장애 시에도 메시지가 **유실되지 않음** (영속성)

**API:** `POST /redis/stream/add`
```json
{
  "content": "예매 요청 내용 (문자열)",
  "metadata": {"user_id": "fan_001", "section": "VIP석", "quantity": 1}
}
```

200건의 예매 요청을 Stream에 추가합니다.

In [ ]:
# Step 1: 200건의 예매 요청을 Redis Stream에 추가
async def add_booking_requests():
    """POST /redis/stream/add 로 200건 큐잉"""
    results = []
    start = time.time()

    async with httpx.AsyncClient(timeout=30.0) as client:
        tasks = []
        for req in requests_data:
            body = {
                "content": f"티켓 예매 요청: {req['user_name']} ({req['request_id']})",
                "metadata": {
                    "request_id": req["request_id"],
                    "user_id": req["user_id"],
                    "user_name": req["user_name"],
                    "requested_section": req["requested_section"],
                    "quantity": req["quantity"],
                    "timestamp": req["timestamp"]
                }
            }
            tasks.append(client.post(f"{BASE_URL}/redis/stream/add", json=body))

        responses = await asyncio.gather(*tasks, return_exceptions=True)

    elapsed = time.time() - start

    success = 0
    for r in responses:
        if not isinstance(r, Exception) and r.status_code == 200:
            success += 1
            results.append(r.json())

    print(f"Stream 추가 완료!")
    print(f"  성공: {success}/{len(requests_data)}건")
    print(f"  소요 시간: {elapsed:.2f}초")
    print(f"  처리량: {success / elapsed:.0f} req/sec")
    if results:
        print(f"\n첫 3개 응답 샘플:")
        for r in results[:3]:
            print(f"  {json.dumps(r, ensure_ascii=False)[:120]}")
    return results

stream_results = await add_booking_requests()
assert len(stream_results) == 200, f"200건 모두 성공해야 합니다 (실제: {len(stream_results)})"
print(f"\n검증 통과: {len(stream_results)}건 모두 Stream에 추가됨")

## Step 2: Consumer Group으로 분산 처리

### 왜 Consumer Group인가?

워커 1개가 200건을 순차 처리하면 너무 느립니다.
Consumer Group을 사용하면 **여러 워커가 메시지를 나눠서 병렬 처리**합니다.

```text
Stream: [req_001] [req_002] [req_003] [req_004] [req_005] [req_006] ...
               │
    Consumer Group: ticket-workers
               │
     ┌─────────┼─────────┐
     ▼         ▼         ▼
  worker-1  worker-2  worker-3
  (req_001) (req_002) (req_003)
  (req_004) (req_005) (req_006)
     ...       ...       ...
```

**핵심:** 같은 메시지는 그룹 내 **하나의 워커만** 받습니다 (중복 처리 방지).

**API:**
- 그룹 생성: `POST /redis/stream/group/create?group=ticket-workers`
- 메시지 읽기: `GET /redis/stream/group/read?group=ticket-workers&consumer=worker-1&count=10`

In [ ]:
# Step 2: Consumer Group 생성 + 3개 워커로 분산 읽기
async def process_with_consumer_group():
    """Consumer Group을 생성하고 3개 워커로 메시지 분산 읽기"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        # 2-1) Consumer Group 생성
        resp = await client.post(
            f"{BASE_URL}/redis/stream/group/create",
            params={"group": "ticket-workers"}
        )
        print(f"Consumer Group 생성: status={resp.status_code}")
        print(f"  응답: {resp.json()}\n")

        # 2-2) 3개 워커로 분산 읽기
        worker_messages = {}  # {worker_name: [messages]}
        all_messages = []

        for worker_name in ["worker-1", "worker-2", "worker-3"]:
            worker_messages[worker_name] = []
            # 각 워커가 여러 번 읽어서 모든 메시지를 소비
            for _ in range(10):  # 최대 10회 반복
                resp = await client.get(
                    f"{BASE_URL}/redis/stream/group/read",
                    params={
                        "group": "ticket-workers",
                        "consumer": worker_name,
                        "count": 10
                    }
                )
                if resp.status_code == 200:
                    data = resp.json()
                    messages = data if isinstance(data, list) else data.get("messages", [])
                    if not messages:
                        break
                    worker_messages[worker_name].extend(messages)
                    all_messages.extend(messages)
                else:
                    break

        print(f"Consumer Group 처리 결과:")
        total = 0
        for wname, msgs in worker_messages.items():
            print(f"  {wname}: {len(msgs)}건 처리")
            total += len(msgs)
        print(f"  합계: {total}건")

        return worker_messages, all_messages

worker_messages, all_consumed = await process_with_consumer_group()
print(f"\n총 소비된 메시지: {len(all_consumed)}건")

## Step 3: Redis Cache로 좌석 관리 (차감 로직)

### 왜 Cache를 사용하는가?

좌석 잔여 수를 **빠르게 조회/갱신**해야 합니다.
Redis Cache는 인메모리이므로 디스크 DB보다 훨씬 빠릅니다.

**좌석 관리 전략:**
```text
Cache Key           Value (dict)                      TTL
────────────────    ────────────────────────────────   ────
seats:VIP석         {"available": 10, "total": 10}     300초
seats:R석           {"available": 15, "total": 15}     300초
seats:S석           {"available": 15, "total": 15}     300초
seats:A석           {"available": 10, "total": 10}     300초
```

**API:**
- 좌석 설정: `POST /redis/cache/set` body `{"key": "seats:VIP석", "value": {"available": 10, "total": 10}, "ttl": 300}`
- 좌석 조회: `GET /redis/cache/get/seats:VIP석` -> `{"value": {...}, "hit": true}`

먼저 좌석을 초기화한 뒤, 200건의 요청을 처리하며 좌석을 차감합니다.

In [ ]:
# Step 3: Redis Cache로 좌석 초기화 + 예매 처리 (차감)
async def manage_seats_with_cache():
    """Cache로 좌석 상태를 관리하고 200건의 요청을 처리"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        # 3-1) 좌석 초기화: 각 섹션별 Cache Set
        print("좌석 초기화 (Cache Set):\n")
        sections_info = {}
        for section in concert["sections"]:
            cache_key = f"seats:{section['name']}"
            cache_body = {
                "key": cache_key,
                "value": {"available": section["seats"], "total": section["seats"]},
                "ttl": 300
            }
            resp = await client.post(f"{BASE_URL}/redis/cache/set", json=cache_body)
            assert resp.status_code == 200, f"Cache Set 실패: {cache_key}"
            sections_info[section["name"]] = section["seats"]
            print(f"  {cache_key} = available:{section['seats']}, total:{section['seats']}")

        # 3-2) 200건 예매 요청 처리 (순서대로)
        print(f"\n예매 처리 시작 (200건)...\n")
        success_list = []
        fail_list = []
        remaining = dict(sections_info)  # 섹션별 잔여석 추적
        start = time.time()

        for req in requests_data:
            section_name = req["requested_section"]
            quantity = req["quantity"]
            cache_key = f"seats:{section_name}"

            # 현재 좌석 조회
            get_resp = await client.get(f"{BASE_URL}/redis/cache/get/{cache_key}")
            get_data = get_resp.json()

            if get_data.get("hit") and get_data["value"]["available"] >= quantity:
                # 차감 가능 -> 좌석 갱신
                new_available = get_data["value"]["available"] - quantity
                total = get_data["value"]["total"]
                set_body = {
                    "key": cache_key,
                    "value": {"available": new_available, "total": total},
                    "ttl": 300
                }
                set_resp = await client.post(f"{BASE_URL}/redis/cache/set", json=set_body)
                assert set_resp.status_code == 200
                remaining[section_name] = new_available
                success_list.append(req)
            else:
                fail_list.append(req)

        elapsed = time.time() - start

        # 3-3) 결과 출력
        print(f"예매 처리 완료!")
        print(f"  성공: {len(success_list)}건")
        print(f"  실패(매진): {len(fail_list)}건")
        print(f"  소요 시간: {elapsed:.2f}초")

        # 3-4) 최종 좌석 상태 확인 (Cache Get)
        print(f"\n최종 좌석 상태 (Cache Get 확인):")
        for section in concert["sections"]:
            cache_key = f"seats:{section['name']}"
            resp = await client.get(f"{BASE_URL}/redis/cache/get/{cache_key}")
            data = resp.json()
            val = data.get("value", {})
            avail = val.get("available", "?")
            total = val.get("total", "?")
            sold = total - avail if isinstance(total, int) and isinstance(avail, int) else "?"
            print(f"  {section['name']}: 잔여 {avail}/{total}석 (판매: {sold}석)  hit={data.get('hit')}")

        # 성공 예매 섹션별 집계
        print(f"\n섹션별 성공 예매 집계:")
        from collections import Counter
        section_counter = Counter(r["requested_section"] for r in success_list)
        total_revenue = 0
        section_prices = {s["name"]: s["price"] for s in concert["sections"]}
        for sec_name, count in section_counter.items():
            qty_sum = sum(r["quantity"] for r in success_list if r["requested_section"] == sec_name)
            revenue = qty_sum * section_prices.get(sec_name, 0)
            total_revenue += revenue
            print(f"  {sec_name}: {count}건 (좌석 {qty_sum}석, 매출 {revenue:,}원)")
        print(f"  총 매출: {total_revenue:,}원")

        return success_list, fail_list

success_bookings, failed_bookings = await manage_seats_with_cache()
print(f"\n검증: 성공 {len(success_bookings)}건 + 실패 {len(failed_bookings)}건 = {len(success_bookings) + len(failed_bookings)}건")

## Step 4: RabbitMQ Fanout으로 예매 성공 알림 브로드캐스트

### 왜 Fanout Exchange인가?

예매 성공 알림을 **이메일 + SMS** 두 채널로 동시에 보내야 합니다.
Fanout Exchange는 메시지를 **바인딩된 모든 큐**에 복사합니다.

```text
예매 성공 이벤트
      │
      │  POST /rabbitmq/fanout/publish
      ▼
┌──────────────────────┐
│   Fanout Exchange    │  ← 모든 큐에 같은 메시지 전달
└──────────────────────┘
      │              │
      ▼              ▼
┌──────────┐  ┌──────────┐
│  Email   │  │   SMS    │
│  Queue   │  │  Queue   │
│ (noti-   │  │ (noti-   │
│  email)  │  │  sms)    │
└──────────┘  └──────────┘
```

**장점:**
- 한 번 발행으로 여러 채널에 동시 전달
- 이메일 서비스가 장애나도 SMS는 정상 발송 (독립 처리)
- 새 채널 추가 시 큐만 바인딩하면 됨 (코드 변경 불필요)

**API:**
- 큐 바인딩: `POST /rabbitmq/fanout/bind?queue_name=notification-email`
- 메시지 발행: `POST /rabbitmq/fanout/publish` body `{"content": "...", "metadata": {...}}`

In [ ]:
# Step 4: Fanout Exchange로 예매 성공 알림 브로드캐스트
async def send_fanout_notifications():
    """예매 성공자에게 이메일+SMS 알림을 Fanout으로 발송"""
    async with httpx.AsyncClient(timeout=30.0) as client:
        # 4-1) 알림 큐 바인딩
        print("Fanout Exchange에 알림 큐 바인딩:\n")
        for queue_name in ["notification-email", "notification-sms"]:
            resp = await client.post(
                f"{BASE_URL}/rabbitmq/fanout/bind",
                params={"queue_name": queue_name}
            )
            assert resp.status_code == 200, f"바인딩 실패: {queue_name}"
            print(f"  {queue_name} 바인딩 완료: {resp.json()}")

        # 4-2) 예매 성공자에게 알림 발송
        print(f"\n예매 성공 알림 발송 중 ({len(success_bookings)}건)...\n")
        start = time.time()
        publish_tasks = []

        for booking in success_bookings:
            body = {
                "content": f"예매 성공: {booking['user_name']}님, {concert['name']}",
                "metadata": {
                    "request_id": booking["request_id"],
                    "user_id": booking["user_id"],
                    "user_name": booking["user_name"],
                    "section": booking["requested_section"],
                    "quantity": booking["quantity"],
                    "concert": concert["name"],
                    "venue": concert["venue"],
                    "date": concert["date"]
                }
            }
            publish_tasks.append(
                client.post(f"{BASE_URL}/rabbitmq/fanout/publish", json=body)
            )

        responses = await asyncio.gather(*publish_tasks, return_exceptions=True)
        elapsed = time.time() - start

        success_count = sum(
            1 for r in responses
            if not isinstance(r, Exception) and r.status_code == 200
        )

        print(f"알림 발송 완료!")
        print(f"  발송 성공: {success_count}/{len(success_bookings)}건")
        print(f"  소요 시간: {elapsed:.2f}초")
        print(f"  처리량: {success_count / elapsed:.0f} msg/sec")
        print(f"\n알림 채널별:")
        print(f"  이메일 큐 (notification-email): {success_count}건")
        print(f"  SMS 큐 (notification-sms): {success_count}건")
        print(f"  총 알림 메시지: {success_count * 2}건 (2개 채널)")

        # 알림 샘플 출력
        print(f"\n알림 샘플 (처음 3명):")
        for i, booking in enumerate(success_bookings[:3]):
            print(f"  {i+1}. {booking['user_name']} ({booking['user_id']})")
            print(f"     {booking['requested_section']} {booking['quantity']}석")
            print(f"     -> 이메일 발송, SMS 발송")

        return success_count

notification_count = await send_fanout_notifications()

## 결과 검증

티켓팅 시스템이 올바르게 동작했는지 종합 검증합니다:

1. **Stream 큐잉**: 200건 모두 Stream에 추가되었는가?
2. **Consumer Group**: 워커들이 메시지를 분산 소비했는가?
3. **좌석 관리**: 총 50석이 정확히 소진되었는가? (과매 없음)
4. **알림 발송**: 예매 성공자 전원에게 알림이 발송되었는가?

In [ ]:
# 종합 결과 검증 및 통계
print("=" * 60)
print("BTS 콘서트 티켓팅 시스템 - 종합 검증")
print("=" * 60)

# 1) Stream 큐잉 검증
print(f"\n[1] Redis Stream 큐잉")
print(f"    요청 추가: {len(stream_results)}/200건")
stream_ok = len(stream_results) == 200
print(f"    결과: {'PASS' if stream_ok else 'FAIL'}")

# 2) Consumer Group 검증
print(f"\n[2] Consumer Group 분산 처리")
for wname, msgs in worker_messages.items():
    print(f"    {wname}: {len(msgs)}건")
total_consumed = sum(len(msgs) for msgs in worker_messages.values())
print(f"    총 소비: {total_consumed}건")

# 3) 좌석 관리 검증
print(f"\n[3] 좌석 관리 (Cache)")
total_sold_seats = sum(r["quantity"] for r in success_bookings)
print(f"    예매 성공: {len(success_bookings)}건 (좌석 {total_sold_seats}석)")
print(f"    예매 실패: {len(failed_bookings)}건")
seats_ok = total_sold_seats <= concert["total_seats"]
print(f"    과매 여부: {'없음 (PASS)' if seats_ok else '과매 발생! (FAIL)'}")

# 4) 알림 검증
print(f"\n[4] Fanout 알림 발송")
print(f"    알림 발송: {notification_count}/{len(success_bookings)}건")
notify_ok = notification_count == len(success_bookings)
print(f"    결과: {'PASS' if notify_ok else 'FAIL'}")

# 최종 판정
print(f"\n{'=' * 60}")
all_pass = stream_ok and seats_ok and notify_ok
if all_pass:
    print("최종 결과: 모든 검증 통과!")
    print("\n성공 요인:")
    print("  - Redis Stream: 200건의 요청을 순서대로 큐잉")
    print("  - Consumer Group: 워커 3개가 병렬로 메시지 소비")
    print("  - Redis Cache: 좌석 잔여석을 정확히 추적/차감")
    print("  - Fanout Exchange: 성공자에게 이메일+SMS 동시 알림")
else:
    print("최종 결과: 일부 검증 실패. 위 결과를 확인하세요.")
print("=" * 60)

## 핵심 포인트 정리

### 1. Redis Stream - 요청 큐잉
- `POST /redis/stream/add` 로 요청을 순서대로 저장
- 메시지 ID가 시간 기반이므로 **선착순 보장**
- 서버 재시작 후에도 메시지 유지 (영속성)

### 2. Consumer Group - 분산 처리
- `POST /redis/stream/group/create` 로 그룹 생성
- `GET /redis/stream/group/read` 로 각 워커가 독립적으로 메시지 소비
- 같은 메시지를 두 워커가 받지 않음 (중복 방지)
- 워커 수를 늘리면 처리량이 선형으로 증가

### 3. Redis Cache - 좌석 상태 관리
- `POST /redis/cache/set` 으로 좌석 상태 저장 (value는 dict)
- `GET /redis/cache/get/{key}` 로 빠른 조회
- TTL 설정으로 자동 만료 (메모리 관리)
- 인메모리이므로 DB 조회 대비 수백 배 빠름

### 4. Fanout Exchange - 알림 브로드캐스트
- `POST /rabbitmq/fanout/bind` 로 큐를 Exchange에 바인딩
- `POST /rabbitmq/fanout/publish` 로 한 번 발행하면 모든 큐에 전달
- 이메일 서비스 장애 시에도 SMS는 정상 발송 (독립성)

### 왜 이 조합인가?

| 구간 | 기술 | 이유 |
|------|------|------|
| 요청 수집 | Redis Stream | 순서 보장, 고속, 영속성 |
| 요청 분산 | Consumer Group | 병렬 처리, 중복 방지 |
| 상태 관리 | Redis Cache | 초고속 읽기/쓰기 |
| 알림 전파 | RabbitMQ Fanout | 다중 채널 브로드캐스트 |

## 확장 아이디어

### 1. 대기열 시스템
```python
# 매진 후 150명을 대기열에 추가
# 취소 발생 시 대기 순서대로 기회 제공
# Redis Sorted Set 활용 (Score = 요청 시각)
```

### 2. 좌석 임시 점유 (10분 TTL)
```python
# 예매 성공 -> 임시 점유 (10분 안에 결제)
# POST /redis/cache/set key="hold:fan_001" value={...} ttl=600
# 결제 완료 -> 확정 / 시간 초과 -> 자동 해제
```

### 3. 실시간 잔여석 알림
```python
# 좌석이 차감될 때마다 Fanout으로 잔여석 브로드캐스트
# content: "잔여석 업데이트"
# metadata: {"VIP석": 3, "R석": 7, "S석": 5, "A석": 2}
```

### 4. Rate Limiting
```python
# 사용자당 요청 제한 (1분에 5회)
# Redis Cache로 요청 횟수 추적
# key="rate:fan_001" value={"count": 3} ttl=60
```

In [ ]:
# 리소스 정리
print("리소스 정리...\n")

# Stream 데이터 확인 (읽기 전용 - Stream은 별도 삭제 API가 없으므로 확인만)
async with httpx.AsyncClient(timeout=10.0) as client:
    # 최종 Stream 상태 읽기
    resp = await client.get(
        f"{BASE_URL}/redis/stream/read",
        params={"count": 5, "last_id": "0"}
    )
    if resp.status_code == 200:
        data = resp.json()
        msg_count = len(data) if isinstance(data, list) else len(data.get("messages", []))
        print(f"  Stream 잔여 메시지 (최근 5건 샘플): {msg_count}건")

    # Cache TTL 확인 (300초 후 자동 만료)
    for section in concert["sections"]:
        cache_key = f"seats:{section['name']}"
        resp = await client.get(f"{BASE_URL}/redis/cache/get/{cache_key}")
        data = resp.json()
        print(f"  Cache {cache_key}: hit={data.get('hit')} (TTL 300초 후 자동 만료)")

print(f"\n최종 요약:")
print(f"  총 요청: {len(requests_data)}건")
print(f"  예매 성공: {len(success_bookings)}건")
print(f"  예매 실패: {len(failed_bookings)}건")
print(f"  알림 발송: {notification_count}건 (x2 채널 = {notification_count * 2}건)")
print(f"\nChallenge 12 완료!")

---

## 네비게이션

[← 이전 (Challenge 11)](./11_challenge_coupang_order.ipynb) | [다음 (Challenge 13) →](./13_challenge_kakaotalk_chat.ipynb)

---

### 학습 체크리스트

- [ ] Redis Stream으로 대량 요청을 순서대로 큐잉하는 방법 이해
- [ ] Consumer Group으로 여러 워커가 병렬 처리하는 원리 이해
- [ ] Redis Cache로 상태를 관리하고 TTL을 설정하는 방법 이해
- [ ] Fanout Exchange로 다중 채널 알림을 브로드캐스트하는 패턴 이해
- [ ] Stream + Consumer Group + Cache + Fanout 조합의 실전 활용 경험